# City of Fairfax, VA — Land Value Tax Model

Revenue-neutral **4:1 split-rate** shift on the **City of Fairfax real-estate levy only**
(independent city, FIPS **51600** — *not* Fairfax County). All existing exemptions preserved.

## Modeling decisions (locked before any code)
1. **Government body** — City of Fairfax independent-city real-estate levy only.
2. **Reform** — split-rate, land taxed at **4×** the improvement rate, revenue-neutral.
3. **Exemptions** — preserved. The assessor's `tax_exempt='Yes'` flag (govt, churches,
   schools, parks, etc.) is held out of the solver and excluded from the export/charts.
4. **Validation control** — the City publishes no single CY2026 taxable-AV line, so we
   reconcile the modeled base against the City's FY2026 budget agenda (each 1¢ of rate
   ⇒ $834,958 ⇒ CY2025 taxable AV $8.3496B), rolled forward by the documented +4.58%
   2026 residential equalization.

## Tax system (City of Fairfax)
- Virginia assesses real property at **100% of fair market value** (Va. Code §58.1-3201;
  confirmed in the City's FY2026 budget public-hearing notice: *"assessment ratio of 100
  percent of fair market value"*).
- **FY2027 real-estate rate = $1.0725 per $100 AV (= 10.725 mills), effective Jan 1, 2026**
  — the rate adopted (May 5, 2026) against the TY2026 reassessment that this data reflects.
  (FY2026 prior rate was $1.055.) The split-rate solver is revenue-neutral, so the rate
  sets the current-tax level and the land/improvement millages are solved to match it.

## Data (cached; City of Fairfax assessment + GIS, TY2026 reassessment)
| File | Provides |
|---|---|
| `data/fairfax_assessments_2026.parquet` | `land_value`, `building_value`, `extra_features_value`, `total_value`, `tax_exempt`, areas |
| `data/fairfax_search_index.parquet` | land-use code `luc` + `luc_description`, `building_type`, `year_built` |
| `data/fairfax_parcels.gpq` | parcel geometry + zoning (`PIN`) |

## Column mapping
| Concept | Column | Notes |
|---|---|---|
| Land value | `land_value` | 100% FMV |
| Improvement value | `building_value` (+ `extra_features_value`, =0 here) | |
| Total value | `total_value` | land + building + extra (exact) |
| Use code | `luc` / `luc_description` | assessor land-use code |
| Exemption flag | `tax_exempt` → `full_exmp` | 'Yes' = fully exempt (176 parcels) |
| Join keys | `account_number` (assess↔search), `parcel_id_detail`↔`PIN` (→geometry) | |

**Sources:** City of Fairfax FY2026 budget agenda item 12a (3/11/2025, fairfax.granicus.com);
City of Fairfax Real Estate Tax & Assessments (fairfaxva.gov); FFXnow / Patch FY2027 budget
coverage (rate $1.0725 effective 1/1/2026, +4.58% residential reassessment).

## Section 1 — Imports and setup

In [ ]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')  # headless
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

# Constants
CITY_NAME = 'fairfax'
STATE_FIPS = '51'
COUNTY_FIPS = '600'             # City of Fairfax independent city (FIPS 51600)
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

ASSESSMENT_RATIO = 1.0          # VA: 100% of fair market value (Va. Code 58.1-3201)
REAL_ESTATE_RATE = 1.0725       # $ per $100 AV — FY2027 adopted, effective Jan 1 2026
MILLAGE = REAL_ESTATE_RATE * 10 # per-$1000 convention used by lvt_utils -> 10.725 mills

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Load and join cached parcel data

The City of Fairfax assessment and GIS data were scraped (TY2026 reassessment) and cached
locally. Three files join into one parcel frame: assessments ← search index on
`account_number` (100% coverage), then geometry by normalized parcel id (≈99.97%).

In [ ]:
assess = pd.read_parquet(DATA_DIR / 'fairfax_assessments_2026.parquet')
search = pd.read_parquet(DATA_DIR / 'fairfax_search_index.parquet')
geom   = gpd.read_parquet(DATA_DIR / 'fairfax_parcels.gpq')

# Assessments <- search index (land-use code) on account_number
df = assess.merge(
    search[['account_number', 'luc', 'luc_description', 'building_type', 'year_built']],
    on='account_number', how='left',
)

# Attach geometry via normalized parcel id (assessment parcel_id_detail <-> geometry PIN)
def _norm(x):
    return ''.join(str(x).split())

df['_pid'] = df['parcel_id_detail'].map(_norm)
geom = geom.copy()
geom['_pid'] = geom['PIN'].map(_norm)
geom_small = geom[['_pid', 'geometry']].drop_duplicates('_pid')

gdf = gpd.GeoDataFrame(
    df.merge(geom_small, on='_pid', how='left'),
    geometry='geometry', crs=geom.crs,
)

print(f"Parcels: {len(gdf):,}")
print(f"With geometry: {gdf.geometry.notna().sum():,}  ({gdf.geometry.notna().mean()*100:.2f}%)")
print(f"luc nulls: {gdf['luc'].isna().sum()}")
print(f"assessment year(s): {sorted(assess['assessment_year'].unique())}")

## Section 3 — Classify parcels and flag exemptions

Exemptions come straight from the assessor's `tax_exempt` flag. Property categories are
mapped from the assessor land-use code (`luc`). Residual buckets (`Other`, `Other
Commercial`, `Other Residential`) stay small and are overwhelmingly exempt civic uses.

In [ ]:
# Full exemption flag — assessor 'tax_exempt' == 'Yes'
gdf['full_exmp'] = gdf['tax_exempt'].eq('Yes').astype(int)

# Property categories from assessor land-use code
def categorize(luc):
    luc = str(luc).strip()
    base = {
        '210': 'Single Family Residential', '211': 'Single Family Residential',
        '270': 'Townhome / Rowhouse',       '280': 'Condominium',
        '220': 'Small Multi-Family (2-4 units)', '352': 'Large Multi-Family (5+ units)',
        '311': 'Other Residential',         '348': 'Other Residential',
        '300': 'Office / Commercial Condo', '344': 'Office / Commercial Condo',
        '353': 'Retail / General Commercial', '412': 'Retail / General Commercial',
        '350': 'Retail / General Commercial', '349': 'Retail / General Commercial',
        '304': 'Retail / General Commercial', '303': 'Retail / General Commercial',
        '465': 'Mixed Use',                 '332': 'Hotel',
        '325': 'Industrial', '390': 'Industrial', '395': 'Industrial',
        '347': 'Transportation - Parking',  '345': 'Transportation - Parking',
        '011': 'Vacant Land', '015': 'Vacant Land', '019': 'Vacant Land',
        '021': 'Vacant Land', '013': 'Vacant Land', '010': 'Vacant Land',
    }
    other_comm = {'324', '327', '326', '423', '410', '528', '329', '330', '381', '426', '342'}
    if luc in base:
        return base[luc]
    if luc in other_comm:
        return 'Other Commercial'
    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf['luc'].map(categorize)

# Override: any $0-improvement parcel is Vacant Land regardless of code
gdf.loc[gdf['building_value'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())
n_ex = int(gdf['full_exmp'].sum())
print(f"\nFully exempt: {n_ex:,} parcels  (${gdf.loc[gdf['full_exmp']==1,'total_value'].sum():,.0f} AV)")

## Section 4 — Current tax model and base reconciliation

Virginia assesses at 100% of fair market value, and `land + building + extra = total`
exactly, so the tax base is the total assessed value at the $1.0725/$100 rate. We then
reconcile the modeled taxable base against the City's published control.

In [ ]:
# 100% FMV base; land + building (+ extra features, =0 here) = total
gdf['taxable_land_value']        = gdf['land_value'].clip(lower=0)
gdf['taxable_improvement_value'] = (gdf['building_value'] + gdf['extra_features_value']).clip(lower=0)
gdf['taxable_value']             = gdf['taxable_land_value'] + gdf['taxable_improvement_value']
gdf['millage_rate']              = MILLAGE  # 10.725 mills = $1.0725 / $100

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

# Base reconciliation against an independent City control.
# City FY2026 budget agenda (3/11/2025): each 1c of the real-estate rate yields $834,958
#   => CY2025 taxable AV = 834,958 / 0.0001 = $8.3496B.
# Our data is the CY2026 (Jan 1 2026) reassessment; 2026 residential equalization +4.58%.
taxable_av = float(gdf.loc[gdf['full_exmp'] == 0, 'taxable_value'].sum())
CY2025_TAXABLE_AV = 834_958 / 0.0001                 # $8.3496B  (primary, City agenda 12a)
roll_fwd = CY2025_TAXABLE_AV * 1.0458                # conservative: residential equalization only

print(f"Modeled current real-estate levy: ${current_revenue:,.0f}  @ ${REAL_ESTATE_RATE}/$100")
print(f"Modeled CY2026 taxable AV:        ${taxable_av:,.0f}")
print(f"City CY2025 taxable AV (control): ${CY2025_TAXABLE_AV:,.0f}"
      f"  -> modeled {(taxable_av/CY2025_TAXABLE_AV-1)*100:+.2f}% (full reassessment growth)")
print(f"Conservative roll-fwd (+4.58%):   ${roll_fwd:,.0f}"
      f"  -> modeled {(taxable_av/roll_fwd-1)*100:+.2f}% (residual = commercial + new construction)")

gap = (taxable_av / roll_fwd - 1) * 100
assert abs(gap) < 5.0, f"Base reconciliation gap {gap:.1f}% exceeds threshold"
print(f"\nBase reconciliation OK: modeled base {gap:+.2f}% vs conservative control.")

## Section 5 — Revenue-neutral 4:1 split-rate

Fully-exempt parcels are held out of the solver and excluded from the export and charts
(they carry no signal). Land is taxed at 4× the improvement millage; total revenue is held
constant.

In [ ]:
# Hold out fully-exempt parcels (excluded from model, export, and charts)
n_exempt = int((gdf['full_exmp'] == 1).sum())
gdf = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, charts).")
print(f"Land millage:        {land_millage:.4f}  (${land_millage/10:.4f}/$100)")
print(f"Improvement millage: {improvement_millage:.4f}  (${improvement_millage/10:.4f}/$100)")
print(f"Ratio: {land_millage/improvement_millage:.2f}:1")
print(f"Revenue neutrality:  new ${new_revenue:,.0f}  vs current ${gdf['current_tax'].sum():,.0f}"
      f"  ({(new_revenue/gdf['current_tax'].sum()-1)*100:+.3f}%)")

category_summary = calculate_category_tax_summary(
    gdf, category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax', new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 4:1 Split-Rate Tax Impact")

### Gate 5 — Artifact scan (read the results)

Interpret the category table against economic priors. Under a split-rate the increase is
bounded by the land-only outcome, so land-heavy categories (vacant, parking) rise and
building-heavy categories (apartments, condos, office) fall. We also confirm residential
land-share genuinely varies (guarding against a flat-ratio artifact à la Maricopa) and that
near-zero-building parcels are genuine land-rich uses, not placeholders.

In [ ]:
d = gdf.copy()
d['bldg_share'] = (d['taxable_improvement_value'] /
                   (d['taxable_land_value'] + d['taxable_improvement_value']).replace(0, np.nan))
scan = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_token_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
).sort_values('median_pct', ascending=False)
print(scan.round(3).to_string())

# Residential land-share variation (a flat assessor ratio would collapse this)
sfr = d[d['PROPERTY_CATEGORY'] == 'Single Family Residential']
ls = sfr['taxable_land_value'] / (sfr['taxable_land_value'] + sfr['taxable_improvement_value'])
print(f"\nSFR land-share: distinct={ls.round(3).nunique()}, "
      f"p10={ls.quantile(.10):.3f}, median={ls.median():.3f}, p90={ls.quantile(.90):.3f}")

# Near-zero-building taxable parcels — confirm they are genuine land-rich uses
tok = d[d['taxable_improvement_value'] <= 1000]
print(f"\nTaxable parcels with <=$1,000 building: {len(tok)}  (${tok['taxable_value'].sum():,.0f} AV)")
print(tok.groupby('luc_description').size().sort_values(ascending=False).to_string())

## Section 6 — Exploration chart (optional)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
med = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
colors = ['#2166ac' if v <= 0 else '#b2182b' for v in med.values]
med.plot.barh(ax=ax, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_title(f'{CITY_NAME.title()} — Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'fairfax_category_preview.png', dpi=150)
plt.close()
print('Saved data/fairfax_category_preview.png')

## Section 7 — Census join + standard export + report

Exact closing pattern. Census join (optional; needs `CENSUS_API_KEY`) must happen before
the export call.

In [ ]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
if not os.getenv('CENSUS_API_KEY'):
    print("Census join: skipped (no CENSUS_API_KEY)")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')
else:
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            try:
                census_data, census_gdf = _future.result(timeout=90)
                gdf = match_to_census_blockgroups(gdf, census_gdf)
                if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                    gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
                if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                    gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
                print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
            except concurrent.futures.TimeoutError:
                print("Census API timed out — skipping census join")
                for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                    gdf[_col] = float('nan')
    except Exception as e:
        print(f"Census join failed: {e}")
        for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
            gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

In [ ]:
# Parcel-map export (ADDED) — does not affect the model, CSV, or report above.
import geopandas as _gpd
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
_src = gdf
if not isinstance(_src, _gpd.GeoDataFrame) or 'geometry' not in _src.columns or _src['geometry'].isna().all():
    _map_gdf = _gpd.GeoDataFrame(_src.copy(), geometry=gdf['geometry'].reindex(_src.index), crs=gdf.crs)
else:
    _map_gdf = _src.copy()
for _c in ['parcel_id_detail', 'location_detail']:
    if _c and _c in gdf.columns:
        _map_gdf[_c] = gdf[_c].reindex(_map_gdf.index)
_map_out = save_parcel_map_export(
    gdf=_map_gdf, city='fairfax',
    output_path='../../analysis/maps/fairfax.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage, improvement_millage=improvement_millage,
    parcel_id_col='parcel_id_detail', parcel_url_template=None,
    owner_name_col=None, owner_address_col='location_detail',
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)
create_parcel_map(_map_out, 'fairfax', simplify_tolerance_m=2)
print('fairfax parcel map done.')

## Validation summary

| Check | Result |
|---|---|
| **Gate 1 — Base reconciliation** | Modeled CY2026 taxable AV **$8.933B**; +2.30% vs. conservative control ($8.732B = City CY2025 base $8.3496B × +4.58% residential equalization). Within tolerance. Revenue-neutral to −0.000%. |
| **Gate 2 — Distribution sanity** | Directions correct: land-heavy categories rise (Parking +91.9%, genuine vacant lots toward the +94.55% ceiling, Retail +31.9%), building-heavy fall (Condominium −28.9%, Large Multi-Family −26.2%, Office −11.2%). No swapped signs. |
| **Gate 3 — Census coverage** | **100.0% matched** (9,682/9,685 have geometry; 3 unmatched parcels lack geometry). |
| **Gate 4 — PNGs** | **7 of 7** (`category_impact`, `distribution`, `ten_pct_share`, + 4 income/minority quintile charts). |
| **Gate 5 — Artifact scan** | **Clean.** Building shares plausible by category (Retail 0.43, Office 0.72, Industrial 0.54, Condo 0.85). No false ceiling-clustering — only all-land uses (parking, vacant) sit near the +94.55% land-only ceiling. The 212 ≤$1,000-building taxable parcels are genuine land-rich uses (HOA common areas, vacant lots, a country-club golf course), **not** Seattle-style placeholders on income property. SFR land-share genuinely varies (346 distinct values, p10–p90 0.29–0.44) ⇒ **not** a Maricopa flat-ratio artifact. |
| Parcel count | 9,509 modeled taxable parcels (+176 fully-exempt held out & excluded from charts; 9,400 in report charts after viz drops 109 zero-value parcels) |
| Millages | Land **20.8655** mills ($2.0865/$100); Improvement **5.2164** mills ($0.5216/$100); 4.00:1 |
| SFR median change | **+0.5%** (near-neutral) |
| Vacant land | mean **+43.3%**, 45.8% of parcels >+10% (median 0% because ~half the category is HOA / floodplain / open-space common land with near-zero value, correctly swept in by the $0-improvement→Vacant rule) |
| Condominium / Large Multi-Family median | **−28.9% / −26.2%** |

### Two results worth explaining (real, not bugs)
1. **Single-family residential is ~neutral (+0.5%), not the usual −5%-to−20%.** The City of
   Fairfax is a high-land-value DC suburb: SFR land share is ~36% (median; p10–p90 0.29–0.44),
   close to the citywide average, so detached homes neither gain nor lose much from the shift.
   This mirrors the Newport News finding. Condos, townhomes and apartments (much higher
   building share) are the residential winners; vacant lots, parking, and strip retail bear
   the increase.
2. **Vacant Land's *median* is 0% while its *mean* is +43%.** The $0-improvement→Vacant rule
   correctly pulls in ~93 HOA common-area parcels plus floodplain/open-space lots that carry
   near-zero land value (current ≈ new ≈ $0 ⇒ 0% change). Genuine *developable* vacant lots
   rise toward the +94.55% land-only ceiling — hence the high mean and that 45.8% of vacant
   parcels see >+10%.

### Limitations
- **Public-service-corporation real estate** (utility/railroad property assessed by the VA
  State Corporation Commission) is not in the local parcel file and is not modeled; the City's
  real-estate levy includes a small PSC component that this base omits.
- A handful of elderly/disabled/disabled-veteran relief recipients are not separately flagged
  per parcel, so their partial relief is not reflected (modeled as fully taxable).
- The City does not publish a single CY2026 taxable-AV line item; the base control is the
  City's CY2025 figure (derived from the FY2026 "$834,958 per 1¢" fiscal-impact statement)
  rolled forward by the documented +4.58% residential equalization.